# YOLO Fire Detector - Cloud Training

Questo notebook esegue la pipeline cloud completa a partire dal bundle del progetto gia' preparato in locale.

## Ordine corretto delle operazioni

1. in locale genera o aggiorna `configs/generated/latest.cloud.yaml`, con il configuratore o a mano

2. in locale esegui `python tools/cloud/prepare_cloud_bundle.py`

3. importa `yolo-fire-detector-cloud.zip` in Colab o in Google Drive

4. esegui le celle in ordine

Il notebook monta Drive, crea i percorsi persistenti e importa il progetto automaticamente; installa le dipendenze e controlla la presenza della GPU; prepara la configurazione finale per la sessione corrente; esegue la pipeline e infine mostra dataset, run ed export finali

Non devi preparare manualmente i percorsi su Google Drive. Una volta fatto il login tutto funzionerà in automatico. Tuttavia, se qualcosa non funziona e vuoi ripartire da capo ti conviene entrare su google drive ed eliminare le cartelle. Inoltre non devi verificare a mano la presenza di `configs/generated/latest.cloud.yaml`: `prepare_cloud_bundle.py` costruisce il bundle proprio a partire da quella configurazione.

## Setup iniziale delle cartelle e dei percorsi


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import yaml

IN_COLAB = "google.colab" in sys.modules
BUNDLE_NAME = "yolo-fire-detector-cloud.zip"
CLOUD_CONFIG_RELATIVE = Path("generated/latest.cloud.yaml")


def find_local_workspace_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "cloud_train.ipynb").exists() and (candidate / "run_experiment.py").exists():
            return candidate
    return current


if IN_COLAB:
    google_colab = __import__("google.colab", fromlist=["drive"])
    google_colab.drive.mount("/content/drive")
    LOCAL_WORKSPACE_ROOT = None
    PERSISTENT_ROOT = Path("/content/drive/MyDrive/yolo-fire-detector")
else:
    LOCAL_WORKSPACE_ROOT = find_local_workspace_root()
    PERSISTENT_ROOT = LOCAL_WORKSPACE_ROOT / "artifacts" / "cloud-notebook-local"

INPUTS_ROOT = PERSISTENT_ROOT / "inputs"
REPO_ROOT = PERSISTENT_ROOT / "repo"
CLOUD_CONFIG_PATH = REPO_ROOT / "configs" / CLOUD_CONFIG_RELATIVE

PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
INPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("IN_COLAB =", IN_COLAB)
print("PERSISTENT_ROOT =", PERSISTENT_ROOT)
print("INPUTS_ROOT =", INPUTS_ROOT)
print("REPO_ROOT =", REPO_ROOT)
print("Config cloud attesa =", CLOUD_CONFIG_PATH)

## Importazione del progetto



Questa cella importa `yolo-fire-detector-cloud.zip`, lo copia nella cache persistente e prepara automaticamente la cartella `repo/` usata dal notebook.



I percorsi persistenti vengono creati dal notebook. La configurazione cloud non deve essere preparata qui: deve essere gia' inclusa nel bundle creato in locale con `prepare_cloud_bundle.py`.


In [ ]:
def find_bundle_candidates() -> list[Path]:
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        if path.exists() and all(existing.resolve() != path.resolve() for existing in candidates):
            candidates.append(path)

    if IN_COLAB:
        add_candidate(Path("/content") / BUNDLE_NAME)
        add_candidate(Path("/content/drive/MyDrive") / BUNDLE_NAME)
    elif LOCAL_WORKSPACE_ROOT is not None:
        add_candidate(LOCAL_WORKSPACE_ROOT / "dist" / BUNDLE_NAME)
        add_candidate(LOCAL_WORKSPACE_ROOT / BUNDLE_NAME)

    add_candidate(INPUTS_ROOT / BUNDLE_NAME)

    return candidates


bundle_candidates = find_bundle_candidates()
selected_bundle = bundle_candidates[0] if bundle_candidates else None

if selected_bundle is None:
    if not IN_COLAB and LOCAL_WORKSPACE_ROOT is not None and (LOCAL_WORKSPACE_ROOT / "run_experiment.py").exists():
        REPO_ROOT = LOCAL_WORKSPACE_ROOT
        CLOUD_CONFIG_PATH = REPO_ROOT / "configs" / CLOUD_CONFIG_RELATIVE
        print("Zip non trovato: uso il repository workspace locale solo per validazione.")
    else:
        raise FileNotFoundError(
            "Bundle non trovato. Carica yolo-fire-detector-cloud.zip in /content o in Google Drive, poi riesegui questa cella."
        )
else:
    persistent_bundle = INPUTS_ROOT / BUNDLE_NAME
    if selected_bundle.resolve() != persistent_bundle.resolve():
        shutil.copy2(selected_bundle, persistent_bundle)
    print("Zip selezionato =", selected_bundle)
    print("Zip persistente =", persistent_bundle)

    temp_extract_dir = PERSISTENT_ROOT / "_incoming_repo"
    shutil.rmtree(temp_extract_dir, ignore_errors=True)
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    temp_extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(persistent_bundle, "r") as archive:
        archive.extractall(temp_extract_dir)

    extracted_files = [child for child in temp_extract_dir.iterdir() if child.is_file()]
    extracted_roots = [child for child in temp_extract_dir.iterdir() if child.is_dir()]
    if extracted_files:
        source_root = temp_extract_dir
    elif len(extracted_roots) == 1:
        source_root = extracted_roots[0]
    else:
        source_root = temp_extract_dir

    shutil.copytree(source_root, REPO_ROOT, dirs_exist_ok=True)
    shutil.rmtree(temp_extract_dir, ignore_errors=True)

CLOUD_CONFIG_PRESENT = CLOUD_CONFIG_PATH.exists()
print("Repo pronta =", REPO_ROOT.exists())
print("Config cloud trovata =", CLOUD_CONFIG_PRESENT)
if CLOUD_CONFIG_PRESENT:
    print("Path config cloud =", CLOUD_CONFIG_PATH)
else:
    print("Config cloud assente: puoi usare la cella speciale subito sotto per caricare un nuovo latest.cloud.yaml.")

## Cella speciale: sostituzione rapida di `latest.cloud.yaml`



Usa questa cella solo se devi cambiare la configurazione della run senza ricreare o ricaricare il bundle.



La cella:



1. carica un file YAML da locale

2. controlla che la configurazione sia valida per il cloud

3. salva un backup della configurazione precedente, se esiste

4. sostituisce `repo/configs/generated/latest.cloud.yaml`



Dopo una sostituzione riuscita, riesegui sempre la cella di preparazione config. Se avevi gia' avviato la pipeline con la configurazione precedente, riesegui anche la cella di esecuzione.


In [ ]:
ENABLE_SPECIAL_CONFIG_REPLACEMENT = False
SPECIAL_CONFIG_REPLACED = False

if not ENABLE_SPECIAL_CONFIG_REPLACEMENT:
    print("Cella speciale disattivata. Lasciala cosi' nel flusso normale.")
    print("Se vuoi usarla, imposta ENABLE_SPECIAL_CONFIG_REPLACEMENT = True e riesegui questa cella.")
else:
    if not IN_COLAB:
        raise RuntimeError("La cella speciale di upload e' pensata per Colab, dove e' disponibile l'upload diretto del file.")

    google_colab_files = __import__("google.colab", fromlist=["files"]).files
    uploaded_files = google_colab_files.upload()
    if not uploaded_files:
        raise RuntimeError("Nessun file caricato.")
    if len(uploaded_files) != 1:
        raise ValueError("Carica esattamente un solo file YAML.")

    uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
    uploaded_text = uploaded_bytes.decode("utf-8")
    uploaded_payload = yaml.safe_load(uploaded_text) or {}
    if not isinstance(uploaded_payload, dict):
        raise ValueError("Il file caricato non contiene una mappa YAML valida.")

    for required_key in ["project", "dataset", "training"]:
        if required_key not in uploaded_payload or not isinstance(uploaded_payload[required_key], dict):
            raise ValueError(f"Config cloud non valida: manca la sezione '{required_key}'.")

    training_cfg = uploaded_payload["training"]
    requested_device = str(training_cfg.get("device", "")).strip().lower()
    require_gpu = bool(training_cfg.get("require_gpu", False))
    if requested_device in {"", "cpu"}:
        raise ValueError("Config cloud non valida: training.device non puo' essere vuoto o 'cpu'.")
    if not require_gpu:
        raise ValueError("Config cloud non valida: training.require_gpu deve essere true.")

    CLOUD_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    backup_path = CLOUD_CONFIG_PATH.with_name("latest.cloud.previous-upload.yaml")
    if CLOUD_CONFIG_PATH.exists():
        shutil.copy2(CLOUD_CONFIG_PATH, backup_path)
        print("Backup config precedente =", backup_path)

    uploaded_cache_path = INPUTS_ROOT / "uploaded-latest.cloud.yaml"
    uploaded_cache_path.write_text(uploaded_text, encoding="utf-8")
    CLOUD_CONFIG_PATH.write_text(uploaded_text, encoding="utf-8")

    CLOUD_CONFIG_PRESENT = True
    SPECIAL_CONFIG_REPLACED = True

    print("Config cloud caricata da =", uploaded_name)
    print("Cache upload =", uploaded_cache_path)
    print("Config cloud sostituita in =", CLOUD_CONFIG_PATH)
    print("\nPrime righe della nuova config:")
    print("\n".join(CLOUD_CONFIG_PATH.read_text(encoding="utf-8").splitlines()[:40]))
    print("\nCelle da rieseguire dopo questa sostituzione:")
    print("- sempre: cella 10")
    print("- se avevi gia' lanciato la pipeline con la vecchia config: cella 12")
    print("- se vuoi aggiornare anche il riepilogo finale dopo la nuova run: cella 14")

## Installazione dipendenze e controllo GPU



Questa cella installa `requirements.txt` e verifica che PyTorch veda correttamente la GPU del runtime.


In [ ]:
from shutil import which

print("Installing requirements from", REPO_ROOT / "requirements.txt")
proc = subprocess.Popen(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt"],
    cwd=REPO_ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, proc.args)

if which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi non disponibile in questa sessione.")

import torch

CUDA_READY = torch.cuda.is_available()
print("Torch version =", torch.__version__)
print("Torch CUDA build =", torch.version.cuda)
print("torch.cuda.is_available() =", CUDA_READY)
print("torch.cuda.device_count() =", torch.cuda.device_count())
if CUDA_READY:
    print("GPU torch visibile =", torch.cuda.get_device_name(0))
else:
    print("ATTENZIONE: CUDA non visibile a PyTorch. La config cloud deve richiedere GPU e la pipeline si fermerà.")


## Preparazione della configurazione finale



Questa cella legge `configs/generated/latest.cloud.yaml` incluso nel bundle (oppure il config caricato nelle cella speciale se lo hai fatto), aggiorna `project.persistent_root`, controlla la policy GPU e riscrive la configurazione pronta per la sessione corrente.


In [ ]:
if not CLOUD_CONFIG_PATH.exists():
    raise FileNotFoundError(
        "Config cloud non trovata in repo/configs/generated/latest.cloud.yaml. Usa la cella speciale di upload oppure rigenera il bundle con una latest.cloud valida."
    )

with open(CLOUD_CONFIG_PATH, "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle) or {}
if not isinstance(config, dict):
    raise ValueError(f"Config cloud non valida: {CLOUD_CONFIG_PATH}")

config.setdefault("project", {})
config.setdefault("training", {})
config["project"]["persistent_root"] = str(PERSISTENT_ROOT)

training_cfg = config["training"]
requested_device = str(training_cfg.get("device", "0")).strip() or "0"
require_gpu = bool(training_cfg.get("require_gpu", True))
training_cfg["device"] = requested_device
training_cfg["require_gpu"] = require_gpu

if requested_device.lower() == "cpu":
    raise ValueError("Config cloud invalida: training.device non puo essere 'cpu'. Rigenera latest.cloud.yaml con runtime cloud.")
if require_gpu and not CUDA_READY:
    raise RuntimeError("GPU obbligatoria ma CUDA non visibile a PyTorch. Controlla il runtime GPU di Colab.")

with open(CLOUD_CONFIG_PATH, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, sort_keys=False, allow_unicode=False)

print("Config cloud pronta =", CLOUD_CONFIG_PATH)
print("\nPrime righe:")
print("\n".join(CLOUD_CONFIG_PATH.read_text(encoding="utf-8").splitlines()[:40]))

## Esecuzione della pipeline



Questa cella avvia `run_experiment.py` usando la configurazione cloud gia' preparata nella cella precedente.


In [ ]:
if not CLOUD_CONFIG_PATH.exists():
    raise FileNotFoundError("Config cloud non trovata. Esegui prima la cella di preparazione config.")

import shlex
from IPython import get_ipython

shell = get_ipython()
if shell is None:
    raise RuntimeError("Shell IPython non disponibile in questa sessione notebook.")

command = (
    f"cd {shlex.quote(str(REPO_ROOT))} && "
    f"{shlex.quote(sys.executable)} -u run_experiment.py --config {shlex.quote(str(CLOUD_CONFIG_PATH))}"
)
exit_code = shell.system(command)
if exit_code not in {0, None}:
    raise RuntimeError(f"Pipeline terminata con codice {exit_code}")


## Output finali



Questa cella mostra dataset, run ed export disponibili. Se trova export validi, crea anche uno zip nella cartella `downloads/`.


In [ ]:
exports_root = PERSISTENT_ROOT / "exports"
datasets_root = PERSISTENT_ROOT / "datasets"
runs_root = PERSISTENT_ROOT / "runs"
download_root = PERSISTENT_ROOT / "downloads"
download_root.mkdir(parents=True, exist_ok=True)
exports_archive = download_root / "yolo-fire-detector-exports.zip"
latest_registry = exports_root / "latest.yaml"

if latest_registry.exists():
    print("Latest export registry:")
    print(latest_registry.read_text(encoding="utf-8"))
else:
    print("Nessun export registrato ancora.")

print("\nExports disponibili:")
if exports_root.exists():
    for path in sorted(exports_root.glob("*")):
        print("-", path)
else:
    print("- cartella exports non trovata")

if exports_root.exists() and any(exports_root.iterdir()):
    if exports_archive.exists():
        exports_archive.unlink()
    shutil.make_archive(str(exports_archive.with_suffix("")), "zip", root_dir=exports_root, base_dir=".")
    print("\nArchivio export pronto:", exports_archive)
    if IN_COLAB:
        print("\nPer scaricarlo in locale:")
        print("from google.colab import files")
        print(f"files.download('{exports_archive}')")

print("\nDatasets disponibili:")
if datasets_root.exists():
    for path in sorted(datasets_root.glob("*")):
        print("-", path)
else:
    print("- cartella datasets non trovata")

print("\nRun disponibili:")
if runs_root.exists():
    for path in sorted(runs_root.glob("*")):
        print("-", path)
else:
    print("- cartella runs non trovata")